# TOX3GNN – Optuna + 30-Run  Experiment
 **T4 GPU**

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Edit this to match where your TOX folder lives in Drive ──────
DRIVE_ROOT = '/content/drive/MyDrive/AUA/Thesis/code/HybridGNN/TOX'

os.makedirs(f'{DRIVE_ROOT}/checkpoints_tox21', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results_tox21',     exist_ok=True)

print('Drive mounted. Root:', DRIVE_ROOT)
print('Contents:', os.listdir(DRIVE_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Root: /content/drive/MyDrive/AUA/Thesis/code/HybridGNN/TOX
Contents: ['tox21_dataset (1).csv', 'tox21_dataset.csv', 'TOX3GNN.py', 'optim_TOX3GNN.py', 'results_tox21', 'feat_corr.py', '__pycache__', 'tox21_analysis.ipynb', 'imputation_checkpoints', 'tox21_imputed.csv', 'utils.py', 'TOX3GNN_experiment.ipynb', 'checkpoints_tox21']


## 2 · Install dependencies

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit', 'optuna'],  check=True)

import torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}')
print('Dependencies ready.')


torch=2.11.0+cu128  cuda=True
Dependencies ready.


## 3 · Imports

In [ ]:
import os, sys, time, random, shutil, warnings, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import optuna

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Linear

from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, GATConv, GINConv, SAGEConv
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from rdkit import Chem

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


Device: cuda


## 4 · Load `utils.py` from Drive
This imports your **exact** implementations — nothing is rewritten here.

In [ ]:
# utils.py must be in DRIVE_ROOT alongside tox21_dataset.csv
sys.path.insert(0, DRIVE_ROOT)

from utils import (
    one_hot_encoding,
    get_atom_features,
    get_bond_features,
    create_pytorch_geometric_graph_data_list_from_smiles_and_labels,
    scaffold_split,
    save_ckp,
    load_ckp,
    optimizer_to,
    round_to_4,
)

# Alias used throughout this notebook
smiles_to_graph_list = create_pytorch_geometric_graph_data_list_from_smiles_and_labels

print('utils.py loaded from Drive — using your exact implementations.')


utils.py loaded from Drive — using your exact implementations.


## 5 · Configuration

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
DATA_CSV       = f'{DRIVE_ROOT}/tox21_dataset.csv'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints_tox21'
RESULTS_DIR    = f'{DRIVE_ROOT}/results_tox21'

# ── Optuna toggle ─────────────────────────────────────────────────────────────
# Set RUN_OPTUNA = True to search for hyperparameters first.
RUN_OPTUNA   = False
N_TRIALS     = 20      # number of Optuna trials (each trains 50 epochs)

# ── Known-best params (used when RUN_OPTUNA = False, or as Optuna fallback) ──
KNOWN_LAYER_TYPES = ['sage', 'gat', 'gin']
KNOWN_HIDDEN = 400
KNOWN_DROPOUT = 0.25
KNOWN_LR = 0.0002
KNOWN_WD = 1e-4
# ── 30-run experiment settings ────────────────────────────────────────────────
N_RUNS = 10
MAX_EPOCHS = 500
EVAL_EVERY = 10
PATIENCE = 20
NUM_GRAPHS_PER_BATCH = 64
USE_SCAFFOLD_SPLIT = True
GLOBAL_SEED = 43
POS_WEIGHT = 6.0      # BCEWithLogitsLoss weight for positive class

print('Config OK')
print(f'  RUN_OPTUNA={RUN_OPTUNA}  N_TRIALS={N_TRIALS}')
print(f'  scaffold_split={USE_SCAFFOLD_SPLIT}  N_RUNS={N_RUNS}  MAX_EPOCHS={MAX_EPOCHS}')


Config OK
  RUN_OPTUNA=False  N_TRIALS=20
  scaffold_split=True  N_RUNS=10  MAX_EPOCHS=500


## 6 · Load dataset & build splits

In [ ]:
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

df = pd.read_csv(DATA_CSV)

# All task columns — everything except smiles/mol_id
task_cols = [col for col in df.columns if col not in ['smiles', 'mol_id']]
NUM_TASKS = len(task_cols)
print(f'Tasks ({NUM_TASKS}): {task_cols}')

# Pre-filter invalid SMILES; keep NaN labels as-is (do NOT dropna)
X_smiles_raw = list(df['smiles'])
X_smiles, y_labels = [], []
for _, row in df.iterrows():
    smi = row['smiles']
    if pd.isna(smi):
        continue
    if Chem.MolFromSmiles(str(smi)) is not None:
        X_smiles.append(smi)
        y_labels.append(row[task_cols].values.astype(float))  # shape (NUM_TASKS,), may have NaN

y_labels = np.array(y_labels)   # shape (N, NUM_TASKS)
print(f'Valid SMILES: {len(X_smiles)} / {len(X_smiles_raw)}')
print(f'Label matrix shape: {y_labels.shape}')
print('Missing per task:')
for i, t in enumerate(task_cols):
    n_miss = np.isnan(y_labels[:, i]).sum()
    print(f'  {t:<15}: {n_miss} missing')

# ── Build graph list — one graph per molecule, y = full task vector ──────────
# utils.py's function expects 1-D y; we build Data objects manually here
# so the graph x/edge_index come from utils helpers but y is our 2D vector.
from rdkit.Chem.rdmolops import GetAdjacencyMatrix
import torch
from torch_geometric.data import Data

ref_mol = Chem.MolFromSmiles('O=O')
N_NODE_FEAT = len(get_atom_features(ref_mol.GetAtomWithIdx(0)))
N_EDGE_FEAT = len(get_bond_features(ref_mol.GetBondBetweenAtoms(0, 1)))

print('Building molecular graphs...')
data_list = []
for smi, y_vec in zip(X_smiles, y_labels):
    mol = Chem.MolFromSmiles(smi)
    n   = mol.GetNumAtoms()
    X   = np.zeros((n, N_NODE_FEAT))
    for atom in mol.GetAtoms():
        X[atom.GetIdx()] = get_atom_features(atom)
    X = torch.tensor(X, dtype=torch.float)
    rows, cols = np.nonzero(GetAdjacencyMatrix(mol))
    E  = torch.stack([torch.from_numpy(rows.astype(np.int64)),
                      torch.from_numpy(cols.astype(np.int64))], dim=0)
    ne = 2 * mol.GetNumBonds()
    EF = np.zeros((ne, N_EDGE_FEAT))
    for k, (i, j) in enumerate(zip(rows, cols)):
        EF[k] = get_bond_features(mol.GetBondBetweenAtoms(int(i), int(j)))
    EF = torch.tensor(EF, dtype=torch.float)
    # y: shape (NUM_TASKS,) — NaN kept as float('nan')
    y_tensor = torch.tensor(y_vec, dtype=torch.float)
    data_list.append(Data(x=X, edge_index=E, edge_attr=EF, y=y_tensor))

assert len(data_list) == len(X_smiles)
print(f'Graph list: {len(data_list)} molecules  |  y shape per graph: {data_list[0].y.shape}')

# ── Fixed scaffold split ──────────────────────────────────────────────────────
if USE_SCAFFOLD_SPLIT:
    train_idx, val_idx, test_idx = scaffold_split(X_smiles, seed=GLOBAL_SEED)
else:
    all_idx = list(range(len(data_list)))
    train_idx, temp = train_test_split(all_idx, test_size=0.2, random_state=GLOBAL_SEED)
    val_idx, test_idx = train_test_split(temp, test_size=0.5, random_state=GLOBAL_SEED)
    print(f'Random split → train:{len(train_idx)} val:{len(val_idx)} test:{len(test_idx)}')

val_loader  = DataLoader([data_list[i] for i in val_idx],
                         batch_size=NUM_GRAPHS_PER_BATCH, shuffle=False, drop_last=False)
test_loader = DataLoader([data_list[i] for i in test_idx],
                         batch_size=NUM_GRAPHS_PER_BATCH, shuffle=False, drop_last=False)
trainval_data = [data_list[i] for i in list(train_idx) + list(val_idx)]
train_data    = [data_list[i] for i in train_idx]

print('Data splits ready.')


## 7 · GNN model

In [ ]:
class GNN(torch.nn.Module):
    def __init__(self, layer_types, hidden_dim, dropout, n_tasks=1):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        in_dim = 79
        for lt in layer_types:
            if lt == 'gcn':
                self.convs.append(GCNConv(in_dim, hidden_dim))
            elif lt == 'gat':
                self.convs.append(GATConv(in_dim, hidden_dim))
            elif lt == 'gin':
                mlp = nn.Sequential(
                    nn.Linear(in_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(),
                    nn.Linear(hidden_dim, hidden_dim), nn.ReLU())
                self.convs.append(GINConv(mlp, eps=0.00005, train_eps=True))
            elif lt == 'sage':
                self.convs.append(SAGEConv(in_dim, hidden_dim))
            else:
                raise ValueError(f'Unknown layer type: {lt}')
            in_dim = hidden_dim
        self.drop = nn.Dropout(p=dropout)
        # n_tasks outputs — one logit per task
        self.out  = Linear(hidden_dim * 2, n_tasks)

    def forward(self, x, edge_index, batch_index):
        for conv in self.convs:
            x = conv(x, edge_index)
            x = torch.tanh(x)
        x = self.drop(x)
        x = torch.cat([gmp(x, batch_index), gap(x, batch_index)], dim=1)
        return self.out(x)   # shape (batch_size, n_tasks)


def multitask_loss(logits, targets, pos_weight_tensor):
    """
    BCE loss over all tasks, masking out NaN labels.
    logits  : (batch, n_tasks)
    targets : (batch, n_tasks)  — may contain NaN
    pos_weight_tensor: (n_tasks,)
    """
    valid_mask = ~torch.isnan(targets)          # (batch, n_tasks) bool
    if valid_mask.sum() == 0:
        return torch.tensor(0.0, requires_grad=True, device=logits.device)
    # Replace NaN with 0 so BCE doesn't error — masked out anyway
    targets_clean = targets.clone()
    targets_clean[~valid_mask] = 0.0
    # Compute per-element BCE with per-task pos_weight
    criterion = torch.nn.BCEWithLogitsLoss(
        pos_weight=pos_weight_tensor, reduction='none')
    loss_all = criterion(logits, targets_clean)  # (batch, n_tasks)
    # Zero out the NaN positions and average over valid only
    loss_all = loss_all * valid_mask.float()
    return loss_all.sum() / valid_mask.float().sum()


def evaluate_auc(model, loader, device, task_cols):
    """
    Mean ROC-AUC across all tasks, skipping tasks where the loader
    has fewer than 2 unique labels (can't compute AUC).
    """
    model.eval()
    all_logits, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch.x.float(), batch.edge_index, batch.batch)
            all_logits.append(logits.cpu().numpy())
            all_targets.append(batch.y.cpu().numpy())
    logits_np  = np.concatenate(all_logits,  axis=0)   # (N, n_tasks)
    targets_np = np.concatenate(all_targets, axis=0)   # (N, n_tasks)

    task_aucs = []
    for t_idx, t_name in enumerate(task_cols):
        y_true = targets_np[:, t_idx]
        y_score = torch.sigmoid(torch.tensor(logits_np[:, t_idx])).numpy()
        valid   = ~np.isnan(y_true)
        if valid.sum() < 2 or len(np.unique(y_true[valid])) < 2:
            continue   # skip tasks with no positive or no negative examples
        task_aucs.append(roc_auc_score(y_true[valid], y_score[valid]))

    return float(np.mean(task_aucs)) if task_aucs else float('nan')


print('GNN and helpers defined.')
print(f'Output layer: {len(task_cols) if "task_cols" in dir() else "n_tasks"} tasks')


## 8 · Optuna hyperparameter search

In [ ]:
# ── Per-task positive weights computed from training labels only ─────────────
# Uses your suggested logic: neg_count / pos_count per task, NaN rows ignored
train_targets = np.array([data_list[i].y.numpy() for i in train_idx])  # (N_train, n_tasks)

pw_list = []
for col in range(NUM_TASKS):
    col_vals  = train_targets[:, col]
    valid     = ~np.isnan(col_vals)
    pos_count = (col_vals[valid] == 1).sum()
    neg_count = (col_vals[valid] == 0).sum()
    weight    = neg_count / max(pos_count, 1) if pos_count > 0 else 1.0
    pw_list.append(weight)
    print(f'  {task_cols[col]:<15}: pos={pos_count}  neg={neg_count}  weight={weight:.2f}')

pos_weight_tensor = torch.tensor(pw_list, dtype=torch.float32).to(device)
print(f'\npos_weight_tensor shape: {pos_weight_tensor.shape}  (one per task)')

# ── Optuna (single-task proxy on SR-ARE for speed; full multi-task too slow) ──
SRATE_IDX = task_cols.index('SR-ARE') if 'SR-ARE' in task_cols else 0

if RUN_OPTUNA:
    optuna_train_loader = DataLoader(
        [data_list[i] for i in train_idx],
        batch_size=NUM_GRAPHS_PER_BATCH, shuffle=True, drop_last=True)

    def objective(trial):
        layer_types = [trial.suggest_categorical(f'layer_{i}', ['gcn','gat','gin','sage'])
                       for i in range(3)]
        hidden_dim  = trial.suggest_int('hidden_dim', 64, 256, step=32)
        dropout     = trial.suggest_float('dropout', 0.0, 0.5)
        lr          = trial.suggest_float('lr', 1e-5, 1e-3, log=True)

        m    = GNN(layer_types, hidden_dim, dropout, n_tasks=NUM_TASKS).to(device)
        opt  = torch.optim.Adam(m.parameters(), lr=lr)
        best_val, patience_ctr = 0.0, 0

        for epoch in range(100):
            m.train()
            for batch in optuna_train_loader:
                batch = batch.to(device)
                opt.zero_grad()
                logits = m(batch.x.float(), batch.edge_index, batch.batch)
                loss   = multitask_loss(logits, batch.y, pos_weight_tensor)
                loss.backward()
                opt.step()

            val_auc = evaluate_auc(m, val_loader, device, task_cols)
            trial.report(val_auc, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
            if val_auc > best_val:
                best_val, patience_ctr = val_auc, 0
            else:
                patience_ctr += 1
                if patience_ctr >= 5:
                    break
        return best_val

    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
    study  = optuna.create_study(direction='maximize', pruner=pruner)
    print(f'Running Optuna ({N_TRIALS} trials)...')
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    best_p           = study.best_trial.params
    best_layer_types = [best_p[f'layer_{i}'] for i in range(3)]
    best_hidden      = best_p['hidden_dim']
    best_dropout     = round_to_4(best_p['dropout'])
    best_lr          = round_to_4(best_p['lr'])
    print(f'Best AUC: {study.best_trial.value:.4f}  layers={best_layer_types}')

    from datetime import datetime
    optuna_path = os.path.join(RESULTS_DIR, f'optuna_results_{datetime.today().strftime("%Y-%m-%d")}.txt')
    with open(optuna_path, 'w') as f:
        f.write('Optuna Study Results\n' + '='*60 + '\n')
        f.write(f'Best AUC: {study.best_trial.value:.6f}\n')
        for k, v in study.best_trial.params.items():
            f.write(f'  {k}: {round_to_4(v)}\n')
        f.write('\nAll trials:\n')
        for rank, t in enumerate(sorted(
            [t for t in study.trials if t.value is not None],
            key=lambda t: t.value, reverse=True), 1):
            f.write(f'{rank}. Trial #{t.number}: AUC={t.value:.6f} | {t.params}\n')
    print(f'Optuna results saved to {optuna_path}')

else:
    best_layer_types = KNOWN_LAYER_TYPES
    best_hidden      = KNOWN_HIDDEN
    best_dropout     = KNOWN_DROPOUT
    best_lr          = KNOWN_LR
    print(f'Using known-best params: {best_layer_types}  hidden={best_hidden}')


## 9 · Save helpers

In [ ]:
def append_run_result(results_dir, name, run_id, best_auc, mod_auc, auc_history):
  """Write per-run metrics (best_auc, mod_auc) to the model txt log and master CSV."""
  os.makedirs(results_dir, exist_ok=True)

  # 1. Per-model text log
  txt_path = os.path.join(results_dir, f'{name}.txt')
  with open(txt_path, 'a') as f:
    f.write(
        f'run:{run_id:02d}  best_auc:{best_auc:.6f}  mod_auc:{mod_auc:.6f}\n'
    )

  # 2. Master CSV logging
  csv_path = os.path.join(results_dir, 'all_runs_summary.csv')
  file_exists = os.path.exists(csv_path)

  # Format history values cleanly
  formatted_auc_history = [f'{auc:.6f}' for auc in auc_history]

  with open(csv_path, 'a', newline='') as f:
    writer = csv.writer(f)

    # If the file is new, include 'mod_auc' in the header
    if not file_exists:
      ep_cols = [f'auc_step_{i+1}' for i in range(len(auc_history))]
      writer.writerow(['model', 'run', 'best_auc', 'mod_auc'] + ep_cols)

    # Write the metrics row
    writer.writerow(
        [name, run_id, f'{best_auc:.6f}', f'{mod_auc:.6f}']
        + formatted_auc_history
    )


## 10 · 30-Run experiment


In [ ]:
NAME      = '_'.join(best_layer_types)
ckpt_dir  = os.path.join(CHECKPOINT_DIR, NAME)
model_dir = os.path.join(CHECKPOINT_DIR, NAME + '_best')
os.makedirs(ckpt_dir,  exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

print(f'Model : {NAME}  |  Tasks: {NUM_TASKS}  |  Runs: {N_RUNS}')

all_run_aucs = []   # mean-across-tasks test AUC per run
experiment_start = time.time()

for run in range(N_RUNS):
    run_seed = run
    torch.manual_seed(run_seed)
    np.random.seed(run_seed)
    random.seed(run_seed)

    model     = GNN(best_layer_types, best_hidden, best_dropout, n_tasks=NUM_TASKS).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=best_lr, weight_decay=KNOWN_WD)

    train_loader = DataLoader(
        train_data,
        batch_size=NUM_GRAPHS_PER_BATCH,
        shuffle=True, drop_last=True,
        generator=torch.Generator().manual_seed(run_seed),
    )

    best_val_auc = 0.0
    patience_ctr = 0
    auc_history  = []
    run_start    = time.time()

    for epoch in range(MAX_EPOCHS):
        model.train()
        for batch in train_loader:
            batch  = batch.to(device)
            optimizer.zero_grad()
            logits = model(batch.x.float(), batch.edge_index, batch.batch)
            loss   = multitask_loss(logits, batch.y, pos_weight_tensor)
            loss.backward()
            optimizer.step()

        if (epoch + 1) % EVAL_EVERY == 0:
            val_auc = evaluate_auc(model, val_loader, device, task_cols)
            auc_history.append(round(val_auc, 6))

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                patience_ctr = 0
                state = {
                    'epoch':      epoch + 1,
                    'state_dict': model.state_dict(),
                    'optimizer':  optimizer.state_dict(),
                    'val_auc':    best_val_auc,
                    'layers':     best_layer_types,
                    'hidden':     best_hidden,
                    'dropout':    best_dropout,
                    'lr':         best_lr,
                    'n_tasks':    NUM_TASKS,
                }
                save_ckp(state, True, ckpt_dir, model_dir,
                         f'model_{NAME}_run{run:02d}_tox21.pt',
                         f'best_model_{NAME}_run{run:02d}_tox21.pt')
            else:
                patience_ctr += 1

            if patience_ctr >= PATIENCE:
                print(f'  Run {run:02d} | Early stop @ ep {epoch+1} | best Val AUC {best_val_auc:.4f}')
                break

    # Load best checkpoint for test evaluation
    best_ckpt = os.path.join(model_dir, f'model_{NAME}_run{run:02d}_tox21.pt')
    if os.path.exists(best_ckpt):
        model, optimizer, _ = load_ckp(best_ckpt, model, optimizer)

    # Per-task test AUC
    model.eval()
    all_logits, all_targets = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            all_logits.append(model(batch.x.float(), batch.edge_index, batch.batch).cpu().numpy())
            all_targets.append(batch.y.cpu().numpy())
    logits_np  = np.concatenate(all_logits,  axis=0)
    targets_np = np.concatenate(all_targets, axis=0)

    per_task_auc = {}
    for t_idx, t_name in enumerate(task_cols):
        y_true  = targets_np[:, t_idx]
        y_score = torch.sigmoid(torch.tensor(logits_np[:, t_idx])).numpy()
        valid   = ~np.isnan(y_true)
        if valid.sum() >= 2 and len(np.unique(y_true[valid])) >= 2:
            per_task_auc[t_name] = round(roc_auc_score(y_true[valid], y_score[valid]), 4)
        else:
            per_task_auc[t_name] = float('nan')

    mean_test_auc = float(np.nanmean(list(per_task_auc.values())))
    all_run_aucs.append(mean_test_auc)

    append_run_result(RESULTS_DIR, NAME, run, mean_test_auc, best_val_auc, auc_history)

    run_mins   = (time.time() - run_start) / 60
    total_mins = (time.time() - experiment_start) / 60
    print(f'Run {run:02d} | MeanTestAUC {mean_test_auc:.4f} | ValAUC {best_val_auc:.4f} | '
          f'{run_mins:.1f}m | total {total_mins:.1f}m')
    for t_name, auc_val in per_task_auc.items():
        print(f'       {t_name:<15}: {auc_val}')

print(f'\n{"="*50}')
print(f'EXPERIMENT COMPLETE — {NAME}')
print(f'  Median Test AUC : {np.median(all_run_aucs):.4f}')
print(f'  Std  Test AUC   : {np.std(all_run_aucs):.4f}')
print(f'  Min  Test AUC   : {np.min(all_run_aucs):.4f}')
print(f'  Max  Test AUC   : {np.max(all_run_aucs):.4f}')
print(f'  Results         : {RESULTS_DIR}')


## 11 · Statistical significance
Run after you have 30-run results for all 4 baselines. Paste the AUC lists below.

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.stats as stats

# Ensure RESULTS_DIR and NAME match your notebook variables
csv_path = os.path.join(RESULTS_DIR, 'all_runs_summary.csv')

if os.path.exists(csv_path):
    df_results = pd.read_csv(csv_path)

    # Extract hybrid model scores using mod_auc
    hybrid_scores = df_results[df_results['model'] == NAME]['mod_auc'].values
    print(f'Hybrid ({NAME}): mean={np.median(hybrid_scores):.4f} ± {np.std(hybrid_scores):.4f}\n')

    header = f'{"Model":<12}  {"Mean":>7}  {"Std":>7}  {"p-value":>12}  {"Significant?":>13}'
    print(header)
    print('-' * len(header))

    models = df_results['model'].unique()
    for m in models:
        if m == NAME:
            continue
        baucs = df_results[df_results['model'] == m]['mod_auc'].values
        if len(baucs) < 2:
            continue
        _, p = stats.mannwhitneyu(hybrid_scores, baucs, alternative='greater')
        sig = 'YES p<0.05' if p < 0.05 else 'no'
        print(f'{m:<12}  {np.median(baucs):>7.4f}  {np.std(baucs):>7.4f}  {p:>12.2e}  {sig:>13}')
else:
    print(f"Summary CSV not found at {csv_path}. Run Cell 10 first.")

## 12 · Results plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot( y=all_run_aucs, ax=axes[0], color='steelblue', width=0.4)
sns.stripplot(y=all_run_aucs, ax=axes[0], color='navy', alpha=0.5, jitter=True)
axes[0].axhline(np.median(all_run_aucs), color='red', ls='--',
                label=f'median={np.median(all_run_aucs):.4f} ± {np.std(all_run_aucs):.4f}')
axes[0].set_title(f'{N_RUNS}-run mean-task AUC distribution\n{NAME}')
axes[0].set_ylabel('Mean ROC-AUC (all tasks)')
axes[0].legend()

summary_df  = pd.read_csv(os.path.join(RESULTS_DIR, 'all_runs_summary.csv'))
hybrid_rows = summary_df[summary_df['model'] == NAME]
# Column name written by append_run_result is 'auc_step_N'
auc_cols    = [c for c in summary_df.columns if c.startswith('auc_step_')]
for _, row in hybrid_rows.iterrows():
    axes[1].plot(range(len(auc_cols)), row[auc_cols].values, alpha=0.25, color='steelblue')
axes[1].set_xlabel(f'Eval checkpoint (every {EVAL_EVERY} epochs)')
axes[1].set_ylabel('Val mean ROC-AUC (all tasks)')
axes[1].set_title(f'Val AUC over training — all {N_RUNS} runs')

plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, f'{NAME}_results.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Saved to {plot_path}')


## 13 · Load a saved checkpoint (optional)
Use this cell to reload the best model from any run — e.g. to run inference or inspect weights.
The checkpoint saved in Cell 10 contains the full model state including which layers were used.


In [ ]:
torch.serialization.add_safe_globals([np._core.multiarray.scalar])

# ── Edit run_id to load whichever run you want ────────────────────────────────
run_id       = 0
load_name    = NAME   # or hardcode e.g. 'sage_gin_sage'
load_model_dir = os.path.join(CHECKPOINT_DIR, load_name + '_best')
ckpt_path    = os.path.join(load_model_dir,
                            f'model_{load_name}_run{run_id:02d}_tox21.pt')

# Reconstruct the model with the same architecture
loaded_model = GNN(best_layer_types, best_hidden, best_dropout).to(device)
loaded_opt   = torch.optim.Adam(loaded_model.parameters(), lr=best_lr)

loaded_model, loaded_opt, start_epoch = load_ckp(ckpt_path, loaded_model, loaded_opt)

# Verify
auc = evaluate_auc(loaded_model, test_loader, device)
print(f'Loaded run {run_id} checkpoint (trained to epoch {start_epoch})')
print(f'Test AUC on reload: {auc:.4f}')